[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/12_Transformer_Sentiment_IMDB.ipynb)

# Transformers — Sentiment Analysis with Self-Attention (IMDB)

**Problem type:** supervised binary text classification (positive vs negative review)

---

## What is a Transformer?

A **Transformer** is a neural-network architecture built entirely on *attention* — no recurrent loops, no convolutions.

### Key ideas

| Concept | Plain English |
|---|---|
| **Self-attention** | Every token looks at every other token in the sequence and decides how much to 'attend' to it. A word like *not* can directly influence the meaning of *good* three words away — no matter how long the sentence. |
| **Multi-head attention** | Run several attention operations in parallel ('heads'), each learning a different relationship (syntax, coreference, sentiment …). Concatenate and project the results. |
| **Positional embeddings** | Because attention has no built-in sense of order, we add a learnable position vector to each token embedding so the model knows *where* a word appears. |
| **Residual + LayerNorm** | Skip connections around each sub-layer stabilise training and let gradients flow cleanly through deep stacks. |

### Why does this matter?

This exact encoder design (stacked Transformer blocks) is the backbone of **BERT**, **RoBERTa**, **GPT** and virtually every large language model in production today. Understanding it at this scale gives you a clear mental model of what those systems do.

---

## Dataset: IMDB Movie Reviews

- **50 000** labeled reviews (25 k train / 25 k test), balanced positive/negative.
- Loaded directly from `tensorflow.keras.datasets.imdb` — no download needed.
- Words are already integer-encoded; we cap the vocabulary at **20 000** words and pad/truncate every review to **200 tokens**.


In [ ]:
# ── Setup: imports & reproducibility ──────────────────────────────────
import os, random
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (works on Colab too)
import matplotlib.pyplot as plt

# Deterministic seeds (best-effort on GPU)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))


In [ ]:
# ── Load IMDB dataset ─────────────────────────────────────────────────
NUM_WORDS  = 20000   # keep only the top-20k most frequent words
MAXLEN     = 200     # pad / truncate every review to this length

# load_data encodes reviews as integer sequences and restricts to the
# top NUM_WORDS most frequent words; rarer words map to index 2 (<UNK>).
(x_train_seq, y_train), (x_test_seq, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=NUM_WORDS
)
print(f'Train samples: {len(x_train_seq):,}   Test samples: {len(x_test_seq):,}')
print(f'Label distribution (train) — 0=neg, 1=pos: {dict(zip(*np.unique(y_train, return_counts=True)))}')

# ── Decode one example ─────────────────────────────────────────────────────
word_index = tf.keras.datasets.imdb.get_word_index()
# The index is offset by 3 (0=pad, 1=start, 2=unknown, 3=unused)
idx_to_word = {v + 3: k for k, v in word_index.items()}
idx_to_word.update({0: '<PAD>', 1: '<START>', 2: '<UNK>', 3: '<UNUSED>'})

def decode_review(encoded):
    return ' '.join(idx_to_word.get(i, '?') for i in encoded)

EXAMPLE_IDX = 0
print(f'\n── Example review (index {EXAMPLE_IDX}) ──')
print(f'Label : {"POSITIVE" if y_train[EXAMPLE_IDX] == 1 else "NEGATIVE"}')
print(f'Length: {len(x_train_seq[EXAMPLE_IDX])} tokens before padding')
print('Text  :', decode_review(x_train_seq[EXAMPLE_IDX])[:500], '...')


In [ ]:
# ── Pad / truncate sequences ──────────────────────────────────────────
x_train = tf.keras.preprocessing.sequence.pad_sequences(
    x_train_seq, maxlen=MAXLEN, padding='post', truncating='post'
)
x_test = tf.keras.preprocessing.sequence.pad_sequences(
    x_test_seq, maxlen=MAXLEN, padding='post', truncating='post'
)
print(f'x_train shape: {x_train.shape}')   # (25000, 200)
print(f'x_test  shape: {x_test.shape}')    # (25000, 200)


## Model Architecture

We build two custom Keras layers:

1. **`TokenAndPositionEmbedding`** — combines a token embedding with a *learned* positional embedding so the model knows token order.
2. **`TransformerBlock`** — one encoder block: Multi-Head Attention → Add & Norm → Feed-Forward (two Dense layers) → Add & Norm.

The classifier head is: `GlobalAveragePooling1D` → `Dropout` → `Dense(1, sigmoid)`.

```
Input (200 tokens)
  └── TokenAndPositionEmbedding
        └── TransformerBlock  [MHA + FFN + residuals]
              └── GlobalAveragePooling1D
                    └── Dropout(0.1)
                          └── Dense(20, relu)
                                └── Dropout(0.1)
                                      └── Dense(1, sigmoid)  ← pos/neg probability
```


In [ ]:
# ── Custom Keras layers ───────────────────────────────────────────────

class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    """Combines token embedding + learned positional embedding."""

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = tf.keras.layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = tf.keras.layers.Embedding(
            input_dim=maxlen, output_dim=embed_dim
        )

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


class TransformerBlock(tf.keras.layers.Layer):
    """One Transformer encoder block: MHA → Add&Norm → FFN → Add&Norm."""

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        # Feed-forward sublayer: expand then project back
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1   = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2   = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Self-attention with residual connection
        attn_output = self.att(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        # Feed-forward with residual connection
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


In [ ]:
# ── Hyper-parameters ──────────────────────────────────────────────────
EMBED_DIM  = 64    # token/position embedding size
NUM_HEADS  = 2     # number of attention heads
FF_DIM     = 128   # inner dimension of feed-forward sublayer
EPOCHS     = 3
BATCH_SIZE = 64

# ── Build model (Functional API) ──────────────────────────────────────────
inputs = tf.keras.Input(shape=(MAXLEN,))                        # (batch, 200)
x = TokenAndPositionEmbedding(MAXLEN, NUM_WORDS, EMBED_DIM)(inputs)
x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)                # collapse sequence dim
x = tf.keras.layers.Dropout(0.1)(x)
x = tf.keras.layers.Dense(20, activation='relu')(x)
x = tf.keras.layers.Dropout(0.1)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)    # P(positive)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────
# With a GPU on Colab this takes ~2-3 min; CPU will be slower.
history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)


In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}  ({test_acc*100:.1f}%)')

# ── Learning curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val accuracy', linestyle='--')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train loss')
axes[1].plot(history.history['val_loss'], label='Val loss', linestyle='--')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/transformer_curves.png', dpi=100)
plt.show()
print('Learning curves saved.')


In [ ]:
# ── Example predictions from the test set ─────────────────────────────
SAMPLE_INDICES = [0, 1, 2, 5, 10]

preds = model.predict(x_test[SAMPLE_INDICES], verbose=0).flatten()

print(f'{'Review (first 120 chars)':<60}  {'True':>8}  {'Pred':>8}  {'P(pos)':>8}')
print('-' * 90)
for i, idx in enumerate(SAMPLE_INDICES):
    text  = decode_review(x_test[idx])[:120].replace('\n', ' ')
    true  = 'POS' if y_test[idx] == 1 else 'NEG'
    prob  = preds[i]
    pred  = 'POS' if prob >= 0.5 else 'NEG'
    mark  = '✓' if true == pred else '✗'
    print(f'{text:<60}  {true:>8}  {pred:>8}  {prob:>7.3f}  {mark}')


## Findings

| Metric | Value |
|---|---|
| Final test accuracy | ~85–88% (varies by run) |
| Epochs | 3 |
| Vocab size | 20 000 |
| Sequence length | 200 tokens |

### What self-attention buys over an RNN

| RNN / LSTM | Transformer (this notebook) |
|---|---|
| Processes tokens **sequentially** — earlier context can 'vanish' | Attends to **all** tokens simultaneously — long-range dependencies are first-class |
| Hard to parallelise during training | Fully parallelisable — much faster on modern hardware |
| Depth limited by vanishing gradients | Residual + LayerNorm allow very deep stacks |

### Scale perspective

This notebook uses **1 Transformer block**, **2 heads**, **embed_dim=64** — a tiny model. Production LLMs stack **dozens of blocks**, use **thousands of heads/dimensions**, and are pre-trained on hundreds of billions of tokens. The architecture is identical; only the scale differs.


## Try it yourself

1. **Add more Transformer blocks** — wrap the `TransformerBlock` call in a loop (`for _ in range(N_BLOCKS)`) to stack multiple encoder layers.
2. **More attention heads** — try `NUM_HEADS = 4` or `8` (must divide `EMBED_DIM` evenly).
3. **Larger embedding** — set `EMBED_DIM = 128` or `256` and `FF_DIM = 512`.
4. **Longer context** — increase `MAXLEN` to `400` or `512` to capture more of each review.
5. **Compare to LSTM** — replace the Transformer block with `tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64))` and compare accuracy and training time.
6. **Pre-trained embeddings** — swap the random `Embedding` for GloVe / FastText weights and see how much pre-training helps.
